# 03. The row count is wrong: debugging a query you did not write

## 📚 What you will be able to do

By the end of this lesson you can:

1. Take a query somebody else wrote, run it, and decide whether to trust the number it returns.
2. **Prove** a number is wrong before you know *why* it is wrong.
3. Find and fix six failures that give you a clean run and a wrong answer: join fan-out, a
   `WHERE` clause that turns a `LEFT JOIN` into an `INNER JOIN`, a date range compared as
   text, `NULL` in a comparison, the wrong `COUNT`, and integer division.
4. Say plainly what a number does **not** cover.

Everything here runs on Python's built-in `sqlite3`. Nothing to install, no server, no
database account. It behaves the same on Windows, on macOS and on Google Colab, and after
the first run it needs no internet.

> **About the numbers in this notebook.** All three tables are built from the row samples
> committed to this repository, not from the full files, so that every student in the room
> reconciles against the *same* control total. The loader prints one honest line per table
> saying which copy it used. The border totals below are therefore **not** the real
> border's totals: the shape of the data is real, the level is a sample.

## 🔗 Where this fits

**Assumes you can already read** `SELECT ... FROM ... JOIN ... ON ... WHERE ... GROUP BY`.
The earlier lessons in this folder cover that. This one assumes it and moves on.

**Why this folder exists at all.** It is not part of the twelve official courses of the
diploma. A scan of 322 real job postings collected for this programme found SQL named in
63 of the 183 Saudi-located postings (34%) and in 21 of the 74 junior-plausible ones (28%)
— roughly four times more often than "deep learning" or "neural network". The README in
this folder records that evidence.

**What *this* lesson is for.** Writing SQL is the part you can practise alone. Reading
somebody else's SQL and deciding whether its answer is true is the part the job actually
pays for. A Riyadh fintech data-analyst posting recorded in that same scan asks for
"complex joins, window functions, and CTEs against a large transactional schema without
hand-holding", and for the "ability to scope a vague business question into something
answerable, and to say clearly when the data can't answer it". The second half of that
sentence is this notebook.

**Feeds into:** every analysis you do after it — the cleaning work in Course 05 Unit 2, the
model-input tables in Course 05 Unit 4, and your final project, each of which begins with a
query whose answer you will have to defend to somebody.

## 🎯 The case: 15,841 results that nobody was told about

Between 25 September and 2 October 2020, Public Health England under-reported 15,841
positive COVID-19 tests. Nothing crashed. Every dashboard rendered. The daily case count
came out lower than the truth and entirely plausible.

The cause, as reported at the time, was that laboratory results were being moved in the old
`.XLS` spreadsheet format, which stops at 65,536 rows; each test result took up several
rows, so a file quietly ran out of room at roughly 1,400 cases. The government's own
description of the incident called it a "file size" issue. The people whose results went
missing were never passed to contact tracers, and press reporting at the time put the
number of contacts not followed up in the tens of thousands.

That is not a SQL bug. It is *this lesson's* bug:

> **A pipeline that ran cleanly and returned a plausible, wrong number, with no control
> total anywhere that would have caught it.**

An error message is a gift. It stops you. A wrong number stops nobody — it gets formatted,
pasted into a deck, and decided on.

---

## Building the practice database

Two real files that already ship with this repository become three SQL tables. Read the
cell: you should be able to say where every column came from.

In [1]:
import pathlib
import sqlite3
import sys

import pandas as pd

# --- find the repository (works from any folder, and on Colab) ------------------------
_here = pathlib.Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "tools" / "data.py").exists()), None)
if _root is None:                      # Colab, or a stray copy of this notebook
    import urllib.request
    pathlib.Path("tools").mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/A-Alwabel/"
            "AI-Diploma-Program/main/tools/data.py", "tools/data.py")
    except Exception as _e:
        raise RuntimeError(
            "Could not find the AI Diploma repository from this folder, and could not "
            "download the data loader either. Open this notebook inside a clone of "
            "https://github.com/A-Alwabel/AI-Diploma-Program, or connect to the internet "
            f"and re-run this cell. (underlying error: {_e})") from None
    _root = pathlib.Path.cwd()
sys.path.insert(0, str(_root))
from tools.data import load

# prefer="sample" pins both tables to the copies committed in the repository, so the
# control totals you are asked to reconcile against are identical on every machine.
border = load("border_crossing_data", prefer="sample")
calls_csv = load("montgomery_911_calls", prefer="sample")

# ":memory:" keeps the database in RAM: nothing is written to disk, and re-running this
# cell gives you a clean database again if an experiment goes wrong.
con = sqlite3.connect(":memory:")

crossings = border[["Port Code", "Date", "Measure", "Value"]].rename(columns={
    "Port Code": "port_code", "Date": "crossing_date",
    "Measure": "measure", "Value": "value"})
crossings.insert(0, "crossing_id", range(1, len(crossings) + 1))
crossings.to_sql("crossings", con, index=False, if_exists="replace")

# This is how a dimension table usually gets built in practice: take the descriptive
# columns out of the fact file and de-duplicate them.
ports = border[["Port Code", "Port Name", "State", "Border", "Location"]].drop_duplicates()
ports = ports.rename(columns={
    "Port Code": "port_code", "Port Name": "port_name",
    "State": "state", "Border": "border", "Location": "location"})
ports.to_sql("ports", con, index=False, if_exists="replace")

calls = pd.DataFrame({
    "ts": calls_csv["timeStamp"].astype(str),
    "township": calls_csv["twp"],
    "zip": calls_csv["zip"].astype("Int64"),          # Int64, not int, so blanks stay NULL
    "title": calls_csv["title"],
    "category": calls_csv["title"].str.split(":").str[0].str.strip(),
    "addr": calls_csv["addr"],
})
calls.insert(0, "call_id", range(1, len(calls) + 1))
calls.to_sql("calls", con, index=False, if_exists="replace")

print(f"\ncrossings : {len(crossings):>7,} rows")
print(f"ports     : {len(ports):>7,} rows")
print(f"calls     : {len(calls):>7,} rows")

border_crossing_data: bundled 43,818-row sample of the 346,733-row original (the full file is on this machine but was not used, because prefer='sample') — every number below is for the sample, not the full file. How it was drawn: stratified by (month, crossing type): all 279 months and all 12 crossing types survive, so the seasonal shape holds, but a 'Value' total is about 13% of the real total.
montgomery_911_calls: bundled 25,000-row sample of the 663,522-row original (the full file is on this machine but was not used, because prefer='sample') — every number below is for the sample, not the full file. How it was drawn: 1 row in every 27, evenly spread, so the sample covers the same 2015-12-10 to 2020-07-29 window with all 100 call types and 68 townships.

crossings :  43,818 rows
ports     :     228 rows
calls     :  25,000 rows


### The three tables

| table | one row is | where it came from |
|---|---|---|
| `crossings` | one month, one port, one kind of traffic | US Bureau of Transportation Statistics border-crossing counts |
| `ports` | a border port and where it is | the descriptive columns of the same file, de-duplicated |
| `calls` | one 911 call | Montgomery County, Pennsylvania emergency call log |

`crossings.crossing_date` is stored exactly as the source file wrote it, as text:
`03/01/2019 12:00:00 AM`. `calls.ts` is stored as text too, in a different format:
`2015-12-10 18:02:38`. Neither is an accident — that is what staging tables look like, and
one of them is a trap you will spring later in this lesson.

The next cell defines the only two helpers this notebook uses: `q(sql)` hands you the result as a DataFrame, and `show(sql)` prints it the way a terminal would. `show` also returns the DataFrame, so a cell that ends with a bare `show(...)` gets a `;` — that is Jupyter's way of saying "do not echo the returned value underneath what I just printed".

In [2]:
def q(sql, params=()):
    """Run SQL against the practice database and hand the result back as a DataFrame."""
    return pd.read_sql_query(sql, con, params=params)


def show(sql, note=None, max_rows=30):
    """Run SQL and print the result the way you would read it in a terminal."""
    df = q(sql)
    if note:
        print(note)
    print(df.head(max_rows).to_string(index=False) if len(df) else "(no rows)")
    if len(df) > max_rows:
        print(f"... {len(df) - max_rows} more row(s)")
    print(f"[{len(df):,} row(s)]")
    return df


def _fmt(x):
    x = float(x)
    return f"{x:,.0f}" if x == int(x) else f"{x:,.2f}"


def your_turn(sql, expect=None):
    """Run the SQL you wrote in the cell above and reconcile it against a control total.

    While the cell still contains the word TODO this prints a reminder instead of raising,
    so a fresh copy of this notebook runs top to bottom for the instructor. `expect` is the
    control total the task gives you - it is not the answer, it is the thing your answer
    has to agree with.
    """
    if not sql.strip() or "TODO" in sql:
        print("Nothing to run yet: replace the TODO in the cell above with your SQL.")
        return
    df = q(sql)
    print(df.to_string(index=False) if len(df) else "(no rows)")
    print(f"[{len(df):,} row(s)]")
    if expect is not None:
        got = float(df.iloc[0, 0]) if df.shape == (1, 1) else float(df.iloc[:, -1].sum())
        ok = abs(got - float(expect)) <= 0.01
        print(f"\ncontrol {_fmt(expect)}  |  your query {_fmt(got)}  |  "
              f"{'MATCH' if ok else 'DOES NOT MATCH - look again'}")


show("SELECT * FROM crossings LIMIT 3")
show("SELECT * FROM ports LIMIT 3")
show("SELECT call_id, ts, township, zip, category, title FROM calls LIMIT 3");

 crossing_id  port_code          crossing_date               measure  value
           1       3322 03/01/2019 12:00:00 AM Truck Containers Full     39
           2        105 03/01/2019 12:00:00 AM     Personal Vehicles   1476
           3       3419 03/01/2019 12:00:00 AM                Trucks    861
[3 row(s)]
 port_code  port_name        state           border                    location
      3322 Del Bonita      Montana US-Canada Border POINT (-112.32481 48.63274)
       105  Vanceboro        Maine US-Canada Border  POINT (-67.42955 45.55984)
      3419   Westhope North Dakota US-Canada Border  POINT (-101.02001 48.9094)
[3 row(s)]
 call_id                  ts     township   zip category                       title
       1 2015-12-10 18:02:38     WHITPAIN 19422      EMS            EMS: HEAD INJURY
       2 2015-12-10 19:08:43     LANSDALE 19446  Traffic Traffic: ROAD OBSTRUCTION -
       3 2015-12-10 20:36:49 LOWER MERION 19004  Traffic Traffic: VEHICLE ACCIDENT -
[3 row(s)]


---

## 🧰 The five questions

You will use these on every query in this notebook, and on every query somebody hands you
for the rest of your career. Learn them as five questions, not as six bug names.

1. **What is the control total?** The same quantity computed a second way, ideally with no
   join at all.
2. **Did the row count change?** `COUNT(*)` before the join and after it. A join from many
   facts to one description must not change it.
3. **Is the join key unique on the "one" side?** `SELECT COUNT(*), COUNT(DISTINCT key) FROM dim`.
   If those two numbers differ, every join to that table multiplies rows.
4. **Does one key survive inspection?** Pick a single id and look at every row it produced.
   Ten seconds of looking beats an hour of theorising.
5. **Does the answer obey something you already know?** Percentages add to 100. A filtered
   total is smaller than an unfiltered one. A report titled "every port" has as many rows
   as there are ports.

Before the hard part, here is the easy part — a query that fails *loudly*.

In [3]:
# `port_code` exists in both tables, and SQLite refuses to guess which one you meant.
con.execute("""
    SELECT port_code, SUM(value)
    FROM crossings JOIN ports ON ports.port_code = crossings.port_code
    GROUP BY port_code
""")

OperationalError: ambiguous column name: port_code

`ambiguous column name: port_code`. That error cost you fifteen seconds and it cost your
employer nothing, because the query never produced a number.

**Nothing else in this notebook will raise.** Every query from here on runs cleanly and
returns an answer that a reasonable person would believe.

---

## Bug 1 — the join that quietly multiplies your rows

**The request.** *"Finance is writing a capacity paper. How many personal vehicles crossed
the border in each US state?"*

**The query you were handed.** It runs. It has been in the team's snippets file for months.

In [4]:
handed_1 = """
SELECT p.state,
       SUM(c.value) AS personal_vehicles
FROM crossings c
JOIN ports p ON p.port_code = c.port_code
WHERE c.measure = 'Personal Vehicles'
GROUP BY p.state
ORDER BY personal_vehicles DESC
"""
by_state = show(handed_1)
print(f"\nadds up to: {by_state['personal_vehicles'].sum():,}")

       state  personal_vehicles
       Texas          229441758
  California          187208478
    New York           60727158
     Arizona           53639006
    Michigan           50658203
  Washington           43133375
       Maine           16757356
     Vermont            7982448
   Minnesota            6232437
  New Mexico            4041384
North Dakota            3861776
     Montana            3611290
       Idaho            1163962
      Alaska             565300
        Ohio                 90
[15 row(s)]

adds up to: 669,024,021


Look at that table as a manager would. Texas first, California second, the northern
border strung out behind them, one tiny entry for Ohio. Fifteen states, a sensible order, a
big round total. There is nothing here to be suspicious of.

So be suspicious anyway. **Question 1: what is the control total?**

In [5]:
control_1 = show("""
SELECT COUNT(*) AS rows_scanned,
       SUM(value) AS personal_vehicles
FROM crossings
WHERE measure = 'Personal Vehicles'
""", note="The same quantity, with no join at all:")

wrong = by_state["personal_vehicles"].sum()
right = control_1["personal_vehicles"].iloc[0]
print(f"\nquery says   {wrong:,}")
print(f"control says {right:,}")
print(f"ratio        {wrong / right:.4f}")

The same quantity, with no join at all:
 rows_scanned  personal_vehicles
         3822          334446087
[1 row(s)]

query says   669,024,021
control says 334,446,087
ratio        2.0004


The query is over by a factor of almost exactly two, and you did not need to understand the
query to find that out. You needed one number computed a second way.

Now find out why. **Question 2: did the row count change?**

In [6]:
show("""
SELECT (SELECT COUNT(*) FROM crossings
        WHERE measure = 'Personal Vehicles')                       AS before_join,
       (SELECT COUNT(*) FROM crossings c JOIN ports p ON p.port_code = c.port_code
        WHERE c.measure = 'Personal Vehicles')                     AS after_join
""");

 before_join  after_join
        3822        7629
[1 row(s)]


3,822 rows went into the join and 7,629 came out. The join **invented** 3,807 rows, and
`SUM` faithfully added up every one of them.

**Question 3: is the join key unique on the "one" side?**

In [7]:
show("SELECT COUNT(*) AS rows_in_ports, COUNT(DISTINCT port_code) AS distinct_ports FROM ports")

show("""
SELECT rows_per_port, COUNT(*) AS how_many_ports
FROM (SELECT port_code, COUNT(*) AS rows_per_port FROM ports GROUP BY port_code)
GROUP BY rows_per_port
ORDER BY rows_per_port
""", note="\nHow many rows does each port have in the ports table?");

 rows_in_ports  distinct_ports
           228             117
[1 row(s)]

How many rows does each port have in the ports table?
 rows_per_port  how_many_ports
             1               7
             2             109
             3               1
[3 row(s)]


There are 117 ports and 228 rows describing them. 109 ports are listed twice and one is
listed three times. `ports` is not a table of ports; it is a table of *port descriptions*,
and the join has no way to know that.

**Question 4: look at one key.**

In [8]:
show("SELECT * FROM ports WHERE port_code = 3015",
     note="Port 3015, in the dimension table:")

show("""
SELECT c.crossing_id, c.port_code, c.measure, c.value, p.port_name, p.location
FROM crossings c
JOIN ports p ON p.port_code = c.port_code
WHERE c.crossing_id = 10
""", note="\nOne single crossing, after the join:");

Port 3015, in the dimension table:
 port_code port_name      state           border                             location
      3015  Boundary Washington US-Canada Border POINT (-117.89655000000002 48.54648)
      3015  Boundary Washington US-Canada Border       POINT (-117.62999999999998 49)
      3015  Boundary Washington US-Canada Border                   POINT (-117.83 49)
[3 row(s)]

One single crossing, after the join:
 crossing_id  port_code                measure  value port_name                             location
          10       3015 Truck Containers Empty     19  Boundary       POINT (-117.62999999999998 49)
          10       3015 Truck Containers Empty     19  Boundary                   POINT (-117.83 49)
          10       3015 Truck Containers Empty     19  Boundary POINT (-117.89655000000002 48.54648)
[3 row(s)]


There it is, and no theory was required. The source file holds three slightly different
coordinate strings for Boundary, Washington, so de-duplicating whole rows — which is how
`ports` was built two cells into this notebook — kept all three. Crossing 10 is one row
recording 19 empty truck containers; after the join it is three rows of 19, and
`SUM(value)` reports 57.

**This is fan-out.** A join from many rows to a key that is not unique multiplies rows
silently. `COUNT(*)`, `SUM()` and `AVG()` are all wrong afterwards; `MAX()` and `MIN()`
survive, which is a good way to fool yourself into thinking the join is fine.

### The fix

The duplication comes from exactly one column. Drop it, and the key becomes unique. A
`WITH` block — a **CTE**, the thing the job postings ask for by name — keeps the repair in
one readable place instead of nesting it inside the `FROM`.

In [9]:
fixed_1 = """
WITH ports_1 AS (
    SELECT DISTINCT port_code, port_name, state, border   -- location dropped: it is what duplicated the key
    FROM ports
)
SELECT p.state,
       SUM(c.value) AS personal_vehicles
FROM crossings c
JOIN ports_1 p ON p.port_code = c.port_code
WHERE c.measure = 'Personal Vehicles'
GROUP BY p.state
ORDER BY personal_vehicles DESC
"""
fixed = show(fixed_1)
print(f"\nadds up to:   {fixed['personal_vehicles'].sum():,}")
print(f"control says: {right:,}")
print("reconciled." if fixed["personal_vehicles"].sum() == right else "STILL WRONG")

show("SELECT COUNT(*) AS rows_in_ports_1, COUNT(DISTINCT port_code) AS distinct_ports "
     "FROM (SELECT DISTINCT port_code, port_name, state, border FROM ports)",
     note="\nAnd the repaired dimension really is one row per port:");

       state  personal_vehicles
       Texas          114720879
  California           93604239
    New York           30363579
     Arizona           26819503
    Michigan           25330524
  Washington           21482226
       Maine            8380998
     Vermont            3991224
   Minnesota            3127693
  New Mexico            2020692
North Dakota            1930888
     Montana            1808921
       Idaho             581981
      Alaska             282650
        Ohio                 90
[15 row(s)]

adds up to:   334,446,087
control says: 334,446,087
reconciled.

And the repaired dimension really is one row per port:
 rows_in_ports_1  distinct_ports
             117             117
[1 row(s)]


In [10]:
comparison = by_state.merge(fixed, on="state", suffixes=("_wrong", "_right"))
comparison["ratio"] = (comparison["personal_vehicles_wrong"]
                       / comparison["personal_vehicles_right"]).round(4)
print(comparison.to_string(index=False))

       state  personal_vehicles_wrong  personal_vehicles_right  ratio
       Texas                229441758                114720879 2.0000
  California                187208478                 93604239 2.0000
    New York                 60727158                 30363579 2.0000
     Arizona                 53639006                 26819503 2.0000
    Michigan                 50658203                 25330524 1.9999
  Washington                 43133375                 21482226 2.0079
       Maine                 16757356                  8380998 1.9994
     Vermont                  7982448                  3991224 2.0000
   Minnesota                  6232437                  3127693 1.9927
  New Mexico                  4041384                  2020692 2.0000
North Dakota                  3861776                  1930888 2.0000
     Montana                  3611290                  1808921 1.9964
       Idaho                  1163962                   581981 2.0000
      Alaska        

Read that comparison carefully, because it is the reason nobody caught this for months.

**The order of the states is identical.** Texas first, California second, all the way down.
Every conclusion anybody drew from the chart was right. Only the *numbers* were wrong — and
numbers are what capacity papers, budgets and headcounts are built from.

And look at Ohio: ratio `1.0`. Ohio's single port, Toledo-Sandusky, happens to have one row
in `ports` rather than two, so it alone was not doubled. That is what fan-out looks like in
the wild — not a clean multiplier you can spot, but a mess that is *mostly* double.

### 🖐 Your turn 1

Write the total number of **`Buses`** that crossed at each **border** (`US-Canada Border`
and `US-Mexico Border`). Two rows out.

Your query must not fan out: the two numbers have to add up to **1,084,852**, which is
`SUM(value) WHERE measure = 'Buses'` with no join at all. Put the total as the **last**
column so the checker can add it up.

In [11]:
my_sql = """
-- TODO: write your query here.
-- Hint: start from the CTE in fixed_1 above, then group by border instead of state.
"""
your_turn(my_sql, expect=1_084_852)

Nothing to run yet: replace the TODO in the cell above with your SQL.


---

## Bug 2 — the `WHERE` clause that deleted your `LEFT JOIN`

**The request.** *"Operations wants **every** port with its truck total, so they can see
which ports handle no trucks at all."*

The word that matters is *every*. Whoever wrote this knew that, and used a `LEFT JOIN`.

In [12]:
handed_2 = """
WITH ports_1 AS (SELECT DISTINCT port_code, port_name, state, border FROM ports)
SELECT p.port_code, p.port_name, p.state,
       SUM(c.value) AS trucks
FROM ports_1 p
LEFT JOIN crossings c ON c.port_code = p.port_code
WHERE c.measure = 'Trucks'
GROUP BY p.port_code, p.port_name, p.state
ORDER BY trucks DESC
"""
trucks = show(handed_2, max_rows=8)

 port_code             port_name      state  trucks
      3801               Detroit   Michigan 4617452
      2304                Laredo      Texas 3835172
       901 Buffalo-Niagara Falls   New York 2859751
      3802            Port Huron   Michigan 2471341
      2506             Otay Mesa California 2114328
      2402               El Paso      Texas 1993457
      2305               Hidalgo      Texas 1645944
      2604               Nogales    Arizona 1110107
... 106 more row(s)
[114 row(s)]


It ran. The totals are not inflated — the CTE from Bug 1 is doing its job. The report looks
finished.

**Question 5: does the answer obey something you already know?** A report titled "every
port" should have one row per port.

In [13]:
show("SELECT COUNT(DISTINCT port_code) AS ports_that_exist FROM ports")
print(f"rows the report returned: {len(trucks)}")

 ports_that_exist
              117
[1 row(s)]
rows the report returned: 114


117 ports exist. The report has 114 rows. Three ports are missing from a report whose
entire purpose was to show ports with no trucks.

Find them.

In [14]:
show("""
WITH ports_1 AS (SELECT DISTINCT port_code, port_name, state, border FROM ports)
SELECT port_code, port_name, state, border
FROM ports_1
WHERE port_code NOT IN (SELECT port_code FROM crossings WHERE measure = 'Trucks')
""", note="Ports with no 'Trucks' row anywhere in the fact table:");

Ports with no 'Trucks' row anywhere in the fact table:
 port_code           port_name      state           border
      2582 Cross Border Xpress California US-Mexico Border
      4105     Toledo-Sandusky       Ohio US-Canada Border
      3814             Algonac   Michigan US-Canada Border
[3 row(s)]


Those three are exactly the ports Operations asked about, and the query deleted them.

### Why

`WHERE` runs **after** the join, on the joined result. For a port with no truck rows, the
`LEFT JOIN` produces one row in which every `c.*` column is `NULL`. Then `WHERE c.measure =
'Trucks'` asks whether `NULL = 'Trucks'`. That is not false — it is `NULL`, which `WHERE`
also throws away.

So a filter on the right-hand table of a `LEFT JOIN` silently converts it into an
`INNER JOIN`. The word `LEFT` stays in the query, doing nothing, which is why this bug
survives code review.

### The fix

Move the condition from `WHERE` into the `ON` clause. `ON` decides which rows *match*;
`WHERE` decides which rows *survive*.

In [15]:
fixed_2 = """
WITH ports_1 AS (SELECT DISTINCT port_code, port_name, state, border FROM ports)
SELECT p.port_code, p.port_name, p.state,
       SUM(c.value)  AS trucks,
       COUNT(*)      AS rows_after_join,
       COUNT(c.value) AS rows_that_matched
FROM ports_1 p
LEFT JOIN crossings c
       ON c.port_code = p.port_code
      AND c.measure   = 'Trucks'        -- a matching rule, not a survival rule
GROUP BY p.port_code, p.port_name, p.state
ORDER BY trucks
"""
all_ports = show(fixed_2, max_rows=6)
print(f"\nrows returned: {len(all_ports)}   ports that exist: 117")

 port_code           port_name      state  trucks  rows_after_join  rows_that_matched
      2582 Cross Border Xpress California     NaN                1                  0
      3814             Algonac   Michigan     NaN                1                  0
      4105     Toledo-Sandusky       Ohio     NaN                1                  0
       706        Cape Vincent   New York     0.0                9                  9
      2410           Boquillas      Texas     0.0                5                  5
      2504          San Ysidro California     0.0               30                 30
... 111 more row(s)
[117 row(s)]

rows returned: 117   ports that exist: 117


117 rows, and the three ports Operations cares about are at the top with `trucks` shown as
`NaN`. That is pandas' rendering of SQL's `NULL`, and here it means "there was nothing to
add up" — which is the honest answer, and the one the report was asked for. Note `rows_after_join = 1` and
`rows_that_matched = 0` for those three: that pair of counts is a compact way to see
whether a `LEFT JOIN` actually matched anything, and you will use it again in Bug 5.

**When is the original right?** When you genuinely want only ports with truck traffic. Then
say so, and write `INNER JOIN`, so the next reader knows it was a decision rather than an
accident.

### 🖐 Your turn 2

Return **one row, one column**: how many of the 117 ports have no `Pedestrians` row at all?
The control answer is **2**. Then change your query to list them, and be ready to say which
states they are in.

In [16]:
my_sql = """
-- TODO: one row, one column, named ports_without_pedestrians.
-- Two routes work: a LEFT JOIN with the condition in ON, or NOT IN. Try the LEFT JOIN one.
"""
your_turn(my_sql, expect=2)

Nothing to run yet: replace the TODO in the cell above with your SQL.


---

## Bug 3 — the date range that returned twenty-four years

**The request.** *"How many border crossings were there in 2018?"*

In [17]:
handed_3 = """
SELECT COUNT(*) AS rows_returned,
       SUM(value) AS crossings_2018
FROM crossings
WHERE crossing_date >= '01/01/2018'
  AND crossing_date <= '12/31/2018'
"""
answer_3 = show(handed_3)

show("SELECT COUNT(*) AS rows_in_whole_table, SUM(value) AS crossings_all_years FROM crossings",
     note="\nThe whole table, all years:");

 rows_returned  crossings_2018
         40267      1166235508
[1 row(s)]

The whole table, all years:
 rows_in_whole_table  crossings_all_years
               43818           1266365234
[1 row(s)]


One year out of twenty-four returned 40,267 of the table's 43,818 rows and 92% of its total.
That is question 5 again: **a filtered total that is nearly the unfiltered total is not a
filtered total.**

Ask the data what years it actually gave you.

In [18]:
years = show("""
SELECT substr(crossing_date, 7, 4) AS year_in_row,
       COUNT(*) AS rows_returned
FROM crossings
WHERE crossing_date >= '01/01/2018'
  AND crossing_date <= '12/31/2018'
GROUP BY year_in_row
ORDER BY year_in_row
""", note="Years present in the result of the '2018' query:", max_rows=30)

Years present in the result of the '2018' query:
year_in_row  rows_returned
       1996           1716
       1997           1716
       1998           1716
       1999           1716
       2000           1716
       2001           1716
       2002           1716
       2003           1848
       2004           1848
       2005           1848
       2006           1848
       2007           1848
       2008           1848
       2009           1848
       2010           1848
       2011           1848
       2012           1848
       2013           1848
       2014           1848
       2015           1848
       2016           1573
       2017           1139
       2018           1215
       2019            304
[24 row(s)]


Twenty-four different years, 1996 to 2019, in a query that filters for 2018.

### Why

SQLite has no date type at all. The documentation is blunt about it: *"SQLite does not have
a storage class set aside for storing dates and/or times."* A date lives in a `TEXT`,
`REAL` or `INTEGER` column and is compared using the rules of whatever type it is stored
as. `crossing_date` is text, so `>=` compares it **character by character**, and the
characters at the front of `MM/DD/YYYY` are the month.

In [19]:
show("""
SELECT '03/01/1996' > '01/01/2019' AS march_1996_is_after_january_2019,
       '2018-03-01' > '2019-01-01' AS same_two_dates_written_iso
""");

 march_1996_is_after_january_2019  same_two_dates_written_iso
                                1                           0
[1 row(s)]


March 1996 sorts *after* January 2019, because `0` and `3` decide it before the year is ever
read. Written the other way round, the same two dates compare correctly.

Here is the same fault wearing a more convincing disguise. This filter returns a *subset*,
which makes it look like it worked:

In [20]:
show("""
SELECT COUNT(*) AS rows_returned,
       COUNT(DISTINCT substr(crossing_date, 7, 4)) AS distinct_years,
       COUNT(DISTINCT substr(crossing_date, 1, 2)) AS distinct_months,
       MIN(substr(crossing_date, 1, 2)) AS earliest_month
FROM crossings
WHERE crossing_date >= '06/01/2018'
""", note="'Everything since June 2018':")

show("""
SELECT substr(crossing_date, 7, 4) AS year_in_row,
       COUNT(DISTINCT substr(crossing_date, 1, 2)) AS months_from_that_year,
       MIN(substr(crossing_date, 1, 2)) AS earliest_month
FROM crossings
WHERE crossing_date >= '06/01/2018'
GROUP BY year_in_row
ORDER BY year_in_row
""", note="\nWhich months of which years actually got through:", max_rows=25);

'Everything since June 2018':
 rows_returned  distinct_years  distinct_months earliest_month
         21717              23                7             06
[1 row(s)]

Which months of which years actually got through:
year_in_row  months_from_that_year earliest_month
       1996                      6             07
       1997                      6             07
       1998                      6             07
       1999                      6             07
       2000                      6             07
       2001                      6             07
       2002                      6             07
       2003                      6             07
       2004                      6             07
       2005                      6             07
       2006                      6             07
       2007                      6             07
       2008                      6             07
       2009                      6             07
       2010                     

21,717 rows, roughly half the table — a believable "last few months" answer. Twenty-three
different years went into it, and the second result says exactly how: every year from 1996
to 2017 contributed **six** months, and 2018 contributed **seven**.

Work through the character comparison and that stops being strange. `07/01/1996` beats
`06/01/2018` on the very first character, because `7` is greater than `6` — the year is
never reached. `06/01/2016` matches `06/01/` and then loses on the year. Every row in this
table is dated on the 1st, so the rule the filter really applied was:

> July to December of **every** year in the table, plus June 2018.

That is not a date range. It is an alphabetical accident that happens to return a subset,
which is what makes it more dangerous than Bug 3's version — a subset looks like it worked.

In [21]:
try:
    con.execute("ALTER TABLE crossings ADD COLUMN month_iso TEXT")
except sqlite3.OperationalError:
    pass                                  # column already added by an earlier run of this cell

con.execute("""
    UPDATE crossings
    SET month_iso = substr(crossing_date, 7, 4) || '-'      -- YYYY
                 || substr(crossing_date, 1, 2) || '-'      -- MM
                 || substr(crossing_date, 4, 2)             -- DD
""")
con.commit()

show("SELECT crossing_date, month_iso FROM crossings LIMIT 3")

show("""
SELECT COUNT(*) AS rows_returned, SUM(value) AS crossings_2018
FROM crossings
WHERE month_iso BETWEEN '2018-01-01' AND '2018-12-31'
""", note="\n2018, asked properly:");

         crossing_date  month_iso
03/01/2019 12:00:00 AM 2019-03-01
03/01/2019 12:00:00 AM 2019-03-01
03/01/2019 12:00:00 AM 2019-03-01
[3 row(s)]

2018, asked properly:
 rows_returned  crossings_2018
          1215        45821222
[1 row(s)]


1,215 rows and 45,821,222 crossings — against the 1,166,235,508 the handed query reported.
The original answer was over by a factor of twenty-five.

### The other date bug: the last day of the range

`crossings` is monthly data, always dated on the first of the month, so a `BETWEEN` on it
is safe. `calls` is not: it records a real time of day. Watch what a January range does to
it.

In [22]:
show("""
SELECT (SELECT COUNT(*) FROM calls
        WHERE ts BETWEEN '2016-01-01' AND '2016-01-31')      AS using_between,
       (SELECT COUNT(*) FROM calls
        WHERE ts >= '2016-01-01' AND ts < '2016-02-01')      AS using_half_open
""")

show("""
SELECT COUNT(*) AS calls_on_31_january, MIN(ts) AS first, MAX(ts) AS last
FROM calls WHERE ts LIKE '2016-01-31%'
""", note="\nWhat BETWEEN dropped:");

 using_between  using_half_open
           483              493
[1 row(s)]

What BETWEEN dropped:
 calls_on_31_january               first                last
                  10 2016-01-31 04:48:51 2016-01-31 21:31:10
[1 row(s)]


Ten calls, every one of them after `2016-01-31 00:00:00`, so every one of them fails
`ts <= '2016-01-31'`. `BETWEEN` lost 2% of January and did not mention it.

**The rule: use a half-open interval.** `ts >= start AND ts < next_start`. It is right for
dates, right for timestamps, right for whatever precision the column turns out to have, and
it does not need you to know whether the column has a time component — which, on a table
you did not build, you usually do not.

### 🖐 Your turn 3

Total crossings in the calendar year **2016**, using `month_iso`, written as a half-open
interval. One row, one column. Control: **44,372,428**.

In [23]:
my_sql = """
-- TODO: one row, one column, named crossings_2016. Use >= and < , not BETWEEN.
"""
your_turn(my_sql, expect=44_372_428)

Nothing to run yet: replace the TODO in the cell above with your SQL.


---

## Bug 4 — `NULL` is not a value, and it is not equal to itself

**The request.** *"How many 911 calls came from somewhere other than zip code 19401?"*

In [24]:
show("""
SELECT (SELECT COUNT(*) FROM calls WHERE zip =  19401) AS inside_19401,
       (SELECT COUNT(*) FROM calls WHERE zip != 19401) AS outside_19401,
       (SELECT COUNT(*) FROM calls)                    AS all_calls
""");

 inside_19401  outside_19401  all_calls
         1794          20181      25000
[1 row(s)]


1,794 inside plus 20,181 outside is 21,975. The table has 25,000 rows. **A row is either
inside 19401 or outside it** — there is no third place to be — so 3,025 calls have gone
somewhere that neither query can see.

In [25]:
show("SELECT COUNT(*) AS calls_with_no_zip FROM calls WHERE zip IS NULL")

show("""
SELECT NULL =  NULL   AS null_equals_null,
       NULL != 19401  AS null_is_not_19401,
       NULL =  19401  AS null_is_19401,
       NULL IS NULL   AS null_is_null
""", note="\nWhat SQL actually thinks about NULL:");

 calls_with_no_zip
              3025
[1 row(s)]

What SQL actually thinks about NULL:
null_equals_null null_is_not_19401 null_is_19401  null_is_null
            None              None          None             1
[1 row(s)]


Three of those four answers are `None` — SQL's `NULL` again. Not true, not false. The
SQLite documentation states the rule directly: *"All operators generally evaluate to NULL
when any operand is NULL."* And `WHERE` keeps a row only when the condition is **true**, so
`NULL` rows are dropped by `zip = 19401` and dropped again by `zip != 19401`.

`NULL` does not mean zero and it does not mean empty. It means *unknown*. Two unknown
values are not equal, because you do not know what either of them is.

Only the `IS` family escapes: the documentation notes that *"It is not possible for an IS or
IS NOT expression to evaluate to NULL."*

### The fix — but first, a decision

Do the 3,025 calls with no zip belong in "calls from outside 19401"? SQL cannot answer that.
*You* have to, and then you have to write down which way you went.

In [26]:
show("""
SELECT (SELECT COUNT(*) FROM calls WHERE zip IS NULL OR zip != 19401) AS unknown_counted_as_outside,
       (SELECT COUNT(*) FROM calls WHERE zip IS NOT 19401)            AS same_thing_shorter,
       (SELECT COUNT(*) FROM calls WHERE zip != 19401)                AS unknown_dropped,
       (SELECT COUNT(*) FROM calls)                                   AS all_calls
""");

 unknown_counted_as_outside  same_thing_shorter  unknown_dropped  all_calls
                      23206               23206            20181      25000
[1 row(s)]


23,206 + 1,794 = 25,000. The books balance.

`IS NOT` is SQLite's null-safe `!=`. Other engines spell it `IS DISTINCT FROM`; check
before you use it at work.

### The same fault, with a bigger blast radius: `NOT IN`

*"How many calls came from a zip code that Lansdale never uses?"*

In [27]:
show("""
SELECT COUNT(*) AS answer
FROM calls
WHERE zip NOT IN (SELECT zip FROM calls WHERE township = 'LANSDALE')
""", note="The handed query:")

show("""
SELECT COUNT(*) AS lansdale_rows,
       COUNT(zip) AS with_a_zip,
       COUNT(*) - COUNT(zip) AS with_no_zip,
       COUNT(DISTINCT zip) AS distinct_zips
FROM calls WHERE township = 'LANSDALE'
""", note="\nWhat is inside that subquery:");

The handed query:
 answer
      0
[1 row(s)]

What is inside that subquery:
 lansdale_rows  with_a_zip  with_no_zip  distinct_zips
           415         401           14              3
[1 row(s)]


Zero. Not "few" — zero, out of 25,000 rows, and no error.

Fourteen Lansdale calls have no zip, so the subquery returns a list containing `NULL`.
SQLite's truth table for `NOT IN` says that when the right-hand side contains `NULL` and the
left operand is not found in it, the result is `NULL` — never true. So no row can ever
satisfy the condition. **A single `NULL` inside a `NOT IN` list silently returns nothing.**

In [28]:
show("""
SELECT (SELECT COUNT(*) FROM calls
        WHERE zip NOT IN (SELECT zip FROM calls
                          WHERE township = 'LANSDALE' AND zip IS NOT NULL)) AS not_in_guarded,
       (SELECT COUNT(*) FROM calls a
        WHERE NOT EXISTS (SELECT 1 FROM calls b
                          WHERE b.township = 'LANSDALE' AND b.zip = a.zip)) AS not_exists
""");

 not_in_guarded  not_exists
          19861       22886
[1 row(s)]


Two fixes, two different answers, and the difference is exactly the 3,025 calls with no zip:
guarded `NOT IN` drops them, `NOT EXISTS` keeps them. Neither is wrong. What *would* be
wrong is reporting one of these numbers without saying which rule you used for the calls
whose zip nobody wrote down.

Prefer `NOT EXISTS` as a habit — it is not fooled by `NULL`, and it is usually the faster
plan on a large table.

### 🖐 Your turn 4

How many calls did **not** come from `LOWER MERION`? Count the calls whose township is
missing as "not Lower Merion". One row, one column. Control: **22,898**.

Then run the naive `township != 'LOWER MERION'` version and be ready to say how many rows
it lost and why that number is so small here.

In [29]:
my_sql = """
-- TODO: one row, one column, named calls_not_lower_merion.
"""
your_turn(my_sql, expect=22_898)

Nothing to run yet: replace the TODO in the cell above with your SQL.


---

## Bug 5 — three counts that are not the same count

**The request.** *"How many zip codes does each township cover?"* Three different `COUNT`
spellings answer three different questions, and only one of them was asked.

In [30]:
show("""
SELECT COUNT(*)             AS count_star,
       COUNT(zip)           AS count_zip,
       COUNT(DISTINCT zip)  AS count_distinct_zip
FROM calls
""", note="On the whole table:")

show("""
SELECT township,
       COUNT(*)            AS calls,
       COUNT(zip)          AS calls_with_a_zip,
       COUNT(DISTINCT zip) AS zip_codes_covered
FROM calls
GROUP BY township
ORDER BY calls DESC
LIMIT 8
""", note="\nBy township:");

On the whole table:
 count_star  count_zip  count_distinct_zip
      25000      21975                  97
[1 row(s)]

By township:
        township  calls  calls_with_a_zip  zip_codes_covered
    LOWER MERION   2102              1830                 14
        ABINGTON   1520              1456                 14
      NORRISTOWN   1515              1445                  3
    UPPER MERION   1405               972                  5
      CHELTENHAM   1161              1014                  9
       POTTSTOWN   1009               986                  3
LOWER PROVIDENCE    875               816                  5
  UPPER MORELAND    858               780                  8
[8 row(s)]


| spelling | what it counts | what it answers |
|---|---|---|
| `COUNT(*)` | rows | how many calls |
| `COUNT(zip)` | rows where `zip` is not `NULL` | how many calls we know the zip of |
| `COUNT(DISTINCT zip)` | different non-`NULL` values | how many zip codes the township covers |

25,000, 21,975 and 97. Hand any of them to somebody who asked "how many zip codes?" and
only the third is an answer.

Look at Norristown: 1,515 calls across **3** zip codes. Upper Merion: 1,405 calls, but only
972 of them have a zip at all. If you had reported `COUNT(zip)` as "calls", Upper Merion
would have looked 30% quieter than it is.

`AVG`, `SUM`, `MIN` and `MAX` skip `NULL` the same way `COUNT(column)` does. That is
convenient right up to the moment your denominator is not the number of rows you thought
it was.

### 🖐 Your turn 5

One row per **category** (`EMS`, `Traffic`, `Fire`), with the zip codes covered and the call
count. Put the call count **last** — the checker adds that column up, and it must come to
**25,000**.

In [31]:
my_sql = """
-- TODO: category, zip_codes_covered, calls  (calls last)
"""
your_turn(my_sql, expect=25_000)

Nothing to run yet: replace the TODO in the cell above with your SQL.


---

## Bug 6 — the percentage that was 50 when it was 50.45

**The request.** *"What share of 911 calls are medical?"*

In [32]:
show("""
SELECT COUNT(CASE WHEN category = 'EMS' THEN 1 END)                 AS ems_calls,
       COUNT(*)                                                     AS all_calls,
       COUNT(CASE WHEN category = 'EMS' THEN 1 END) / COUNT(*)       AS share,
       100 * COUNT(CASE WHEN category = 'EMS' THEN 1 END) / COUNT(*) AS percent
FROM calls
""");

 ems_calls  all_calls  share  percent
     12613      25000      0       50
[1 row(s)]


`share` is `0`. That one is harmless: it is so obviously wrong that you fix it in five
seconds.

`percent` is `50`, and *that* is the dangerous one. It is nearly right, it is exactly the
kind of number that goes straight into a slide, and it is wrong.

SQLite's rule: *"Integer divide yields an integer result, truncated toward zero."*
`12613 / 25000` is `0`. `1261300 / 25000` is `50`, not `50.452`. Nothing is rounded —
the remainder is thrown away.

**Question 5 catches it without any of that theory.** Percentages add to 100.

In [33]:
show("""
SELECT category,
       COUNT(*)                                        AS calls,
       100 * COUNT(*) / (SELECT COUNT(*) FROM calls)    AS integer_percent,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM calls), 2) AS real_percent
FROM calls
GROUP BY category
ORDER BY calls DESC
""");

category  calls  integer_percent  real_percent
     EMS  12613               50         50.45
 Traffic   8678               34         34.71
    Fire   3709               14         14.84
[3 row(s)]


50 + 34 + 14 = 98. Two per cent of the calls fell off the table, and the only way to notice
was to add the column up.

### The fix

Make one side of the division a real number. `100.0` instead of `100` is enough; `CAST(x AS
REAL)` is clearer when the literal is not obvious. Then round on purpose, at the end,
rather than by accident, in the middle.

### 🖐 Your turn 6

The share of all calls that are `Fire`, as a percentage with two decimal places. One row,
one column. Control: **14.84**.

In [34]:
my_sql = """
-- TODO: one row, one column, named fire_percent.
"""
your_turn(my_sql, expect=14.84)

Nothing to run yet: replace the TODO in the cell above with your SQL.


---

## 🖐 The audit

This is the whole job in one exercise.

**The situation.** A colleague has left. Their last report is scheduled to run on Sunday and
go to the regional director. You have been asked to sign it off. It runs without error.

**The report.** *"Truck crossings by state in 2018, and each state's share of the total."*

In [35]:
audit_sql = """
SELECT p.state,
       SUM(c.value) AS trucks_2018,
       100 * SUM(c.value) / (SELECT SUM(value) FROM crossings WHERE measure = 'Trucks')
            AS pct_of_total
FROM crossings c
JOIN ports p ON p.port_code = c.port_code
WHERE c.measure = 'Trucks'
  AND c.crossing_date >= '01/01/2018'
  AND c.crossing_date <= '12/31/2018'
GROUP BY p.state
ORDER BY trucks_2018 DESC
"""
audit = show(audit_sql)
print(f"\ntrucks_2018 adds up to : {audit['trucks_2018'].sum():,}")
print(f"pct_of_total adds up to: {audit['pct_of_total'].sum()}")

       state  trucks_2018  pct_of_total
       Texas     16762436            53
    Michigan     12948242            41
    New York      9107628            28
  California      5768552            18
  Washington      3453192            10
     Arizona      2408442             7
       Maine      2228749             7
North Dakota      1720594             5
     Vermont      1473384             4
     Montana      1101787             3
   Minnesota       429649             1
  New Mexico       411860             1
       Idaho       339166             1
      Alaska        58962             0
[14 row(s)]

trucks_2018 adds up to : 58,212,643
pct_of_total adds up to: 179


The percentages add up to 179.

**What you have to do.**

1. Find at least **three** distinct bugs in that query. There are four things wrong with it.
2. For each one, write the diagnostic query that *proves* it — not an argument, a number.
3. Write the corrected report.
4. Reconcile it. The true 2018 truck total, computed with no join at all, is **1,366,522**
   across 144 fact rows. Your `trucks_2018` column must add up to exactly that, and your
   percentage column must add up to 100 (99.99 after rounding is fine — say so rather than
   hiding it).
5. Answer one question in writing: **does the state ranking change?** If it does, name the
   state that moves and say what decision that ranking was going to be used for.

In [36]:
# Diagnostics. One query per bug you suspect. Nothing here has to be pretty.
diagnostic = """
-- TODO: prove bug 1 with a number
"""
your_turn(diagnostic)

Nothing to run yet: replace the TODO in the cell above with your SQL.


In [37]:
# The corrected report. Columns: state, pct_of_total, trucks_2018  (trucks_2018 LAST).
my_report = """
-- TODO: your corrected query
"""
your_turn(my_report, expect=1_366_522)

Nothing to run yet: replace the TODO in the cell above with your SQL.


---

## 💬 Discuss

No single right answer to any of these. Take a side and defend it.

1. **The fan-out fix threw a column away.** Another team would repair `ports` by keeping one
   location per port with `MIN(location)`. A third would keep both rows and add an
   `is_current` flag. Which would you defend to the person who owns that table — and what
   does each choice cost on the day a port genuinely moves?

2. **You find Bug 1 an hour after the number went into a slide the director has already
   presented.** What do you send, to whom, and what exactly do you say about the size of the
   error while you are still not certain of the corrected figure? Where is the line between
   "tell them now" and "be sure first"?

3. **The 3,025 calls with no zip.** Give one use of "calls from outside 19401" where
   including them is the right call, and one where it is clearly wrong. Then say how a
   reader of your report is supposed to know which one you did.

4. **Every fix in this notebook made the query longer.** When is a long, slow, obviously
   correct query better than a short one, and when is that a bad trade? Does your answer
   change if the query is going to run every night for three years with nobody watching?

5. **Which of the five questions would you keep** if you were allowed only one before
   sending a number to somebody who will act on it?

## ⚠️ Where this breaks

**The control-total method needs a control.** Every proof in this notebook worked because a
second, simpler way of getting the number existed. When nobody has an independent count —
a new pipeline, a table only you use — reconciliation is not available. All you have left is
internal consistency: percentages that sum, subtotals that match totals, row counts that
match a source system. Build those checks *while* you build the query, because afterwards
you will not be able to.

**"Did the row count change?" does not catch everything.** A join that drops 100 rows and
duplicates 100 others leaves `COUNT(*)` unchanged and `SUM()` wrong. The row count is a
smoke alarm, not a fire inspection.

**These numbers are a sample's numbers.** Both tables are the row samples committed to this
repository. The border sample's `Value` total is roughly 13% of the real file's, and the
911 sample is one row in twenty-seven. A sample can hide a bug that only bites at full
scale, and it can invent one that disappears at full scale.

**SQLite is more forgiving than the database at your job.** It let you store dates as
text without complaint — that was Bug 3. It also allows this:

In [38]:
show("""
SELECT township, title, COUNT(*) AS calls
FROM calls
GROUP BY township
ORDER BY calls DESC
LIMIT 3
""", note="`title` is neither grouped nor aggregated. SQLite picks one arbitrarily:");

`title` is neither grouped nor aggregated. SQLite picks one arbitrarily:
    township                       title  calls
LOWER MERION Traffic: VEHICLE ACCIDENT -   2102
    ABINGTON      EMS: CARDIAC EMERGENCY   1520
  NORRISTOWN         EMS: ASSAULT VICTIM   1515
[3 row(s)]


SQLite documents that permission under the heading *"Bare columns in an aggregate query"*.
PostgreSQL rejects the same query: its manual states that *"When `GROUP BY` is present, or
any aggregate functions are present, it is not valid for the `SELECT` list expressions to
refer to ungrouped columns except within aggregate functions or when the ungrouped column
is functionally dependent on the grouped columns"*. So a query that runs here can fail
there — and worse, a query that runs on both can mean different things on each. When you
arrive at a job, find out which engine you are on before you trust a habit you learned in
this notebook.

**And the one this whole lesson cannot fix.** Every bug here produced a wrong answer to the
*right* question. Nothing in SQL will tell you that you answered the wrong question, or that
the data cannot answer the question at all. That is the sentence the fintech posting quoted
at the top asks for — *"to say clearly when the data can't answer it"* — and it is a
judgement, not a query.

## The card to keep

Before any number leaves your hands:

1. **Control total.** Compute it a second way, with no join.
2. **Row count** before the join and after it.
3. **`COUNT(*)` vs `COUNT(DISTINCT key)`** on every table you joined *to*.
4. **One key**, inspected by eye.
5. **An invariant**: percentages sum to 100, subtotals sum to the total, "every port" has
   117 rows.

And two habits that remove whole classes of bug before they start:

- Dates in ISO (`YYYY-MM-DD`), compared with `>= start AND < next_start`.
- `NULL` decided on purpose: `IS NULL` / `IS NOT` / `NOT EXISTS`, never a bare `!=`.

## References

Each of these is used above, not decoration.

- SQLite, *Datatypes In SQLite*, §2.2 "Date and Time Datatype" — the source of Bug 3:
  "SQLite does not have a storage class set aside for storing dates and/or times."
  <https://www.sqlite.org/datatype3.html>
- SQLite, *SQL Language Expressions* — the `IS` / `IS NOT` rule and the `NOT IN` truth table
  used in Bug 4, and "Integer divide yields an integer result, truncated toward zero." used
  in Bug 6. <https://www.sqlite.org/lang_expr.html>
- SQLite, *SELECT*, §2.5 "Bare columns in an aggregate query" — the permission relied on in
  the last demonstration of "Where this breaks". <https://www.sqlite.org/lang_select.html>
- PostgreSQL, *SELECT*, "GROUP BY Clause" — the ungrouped-column rule quoted in
  "Where this breaks". <https://www.postgresql.org/docs/current/sql-select.html>
- The Register, "What a Hancock-up: Excel spreadsheet blunder blamed after England
  under-reports 16,000 COVID-19 cases", 5 October 2020 — the 15,841 under-reported
  results, the dates, the `.XLS` 65,536-row limit and the ~1,400 cases per file.
  <https://www.theregister.com/2020/10/05/excel_england_coronavirus_contact_error/>
- Data sources: US Bureau of Transportation Statistics border-crossing counts, and the
  Montgomery County (PA) 911 call log. Both are described in `Course 04/datasets/DATA.md`.